In [ ]:
# ============================================================
# DIRISA SDC 2026 - Teams Qualification
# Notebook 02: Merging + Feature Engineering
# Scope: Eastern Cape (pilot province)
# Strategy: Hybrid (municipality for widest gap, district for trend)
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("/content/drive/MyDrive/dirisa-sdc/processed")
PROCESSED = Path("/content/drive/MyDrive/dirisa-sdc/processed")

In [ ]:
master   = pd.read_csv(DATA / "clean_master_municipality.csv")
youth    = pd.read_csv(DATA / "clean_youth_registration.csv")
district = pd.read_csv(DATA / "clean_district_youth_history.csv")

print("master:", master.shape)
print("youth:", youth.shape)
print("district:", district.shape)

master: (33, 20)
youth: (33, 4)
district: (12, 11)


In [ ]:
youth_slim = youth[["mdb_code", "registered_youth_total"]].copy()
merged = master.merge(youth_slim, on="mdb_code", how="inner")

print("Merged shape:", merged.shape)
print("Columns:", merged.columns.tolist())
print()
print(merged.head(3))

Merged shape: (33, 21)
Columns: ['mdb_code', 'municipality', 'district', 'total_pop_2022', 'youth_pop_20_34_2022', 'youth_share_pct_2022', 'registered_2016', 'valid_votes_2016', 'spoilt_votes_2016', 'votes_cast_2016', 'turnout_pct_2016', 'spoilt_rate_pct_2016', 'registered_2021', 'valid_votes_2021', 'spoilt_votes_2021', 'votes_cast_2021', 'turnout_pct_2021', 'spoilt_rate_pct_2021', 'registered_growth_2016_2021_pct', 'turnout_change_2016_2021_pp', 'registered_youth_total']

  mdb_code      municipality        district  total_pop_2022  \
0    EC124         Amahlathi        Amathole        115703.0   
1    EC102  Blue Crane Route  Sarah Baartman         49883.0   
2      BUF      Buffalo City    Buffalo City        975249.0   

   youth_pop_20_34_2022  youth_share_pct_2022  registered_2016  \
0               24608.0             21.268247          52945.0   
1               11309.0             22.671050          18372.0   
2              238729.0             24.478774         419044.0   



In [ ]:
merged["youth_gap_2026"] = (
    (merged["youth_pop_20_34_2022"] - merged["registered_youth_total"])
    / merged["youth_pop_20_34_2022"]
)

# Clip negatives at 0 (age-band mismatch artefact)
merged["youth_gap_2026"] = merged["youth_gap_2026"].clip(lower=0)

merged["youth_registration_rate_2026"] = (
    merged["registered_youth_total"] / merged["youth_pop_20_34_2022"]
)

print("Municipality features computed.")
print(merged[["municipality", "youth_pop_20_34_2022",
              "registered_youth_total", "youth_gap_2026",
              "youth_registration_rate_2026"]].head(10))

Municipality features computed.
       municipality  youth_pop_20_34_2022  registered_youth_total  \
0         Amahlathi               24608.0                 18461.0   
1  Blue Crane Route               11309.0                  7445.0   
2      Buffalo City              238729.0                177827.0   
3   Dr Beyers Naude               22376.0                 17751.0   
4          Elundini               30752.0                 26517.0   
5        Emalahleni               24376.0                 23549.0   
6           Engcobo               26398.0                 29638.0   
7     Enoch Mgijima               68237.0                 49366.0   
8         Great Kei                7201.0                  8047.0   
9     Intsika Yethu               23324.0                 29190.0   

   youth_gap_2026  youth_registration_rate_2026  
0        0.249797                      0.750203  
1        0.341675                      0.658325  
2        0.255109                      0.744891  
3       

In [ ]:
widest = (
    merged
    .sort_values("youth_gap_2026", ascending=False)
    [["mdb_code", "municipality", "district",
      "youth_pop_20_34_2022", "registered_youth_total",
      "youth_gap_2026", "youth_registration_rate_2026"]]
    .reset_index(drop=True)
)

widest["gap_rank"] = widest.index + 1

print("Top 10 municipalities by widest youth gap:")
print(widest.head(10).to_string(index=False))

Top 10 municipalities by widest youth gap:
mdb_code         municipality       district  youth_pop_20_34_2022  registered_youth_total  youth_gap_2026  youth_registration_rate_2026  gap_rank
   EC102     Blue Crane Route Sarah Baartman               11309.0                  7445.0        0.341675                      0.658325         1
   EC105              Ndlambe Sarah Baartman               18309.0                 12919.0        0.294391                      0.705609         2
   EC145        Walter Sisulu      Joe Gqabi               25164.0                 17836.0        0.291210                      0.708790         3
   EC139        Enoch Mgijima     Chris Hani               68237.0                 49366.0        0.276551                      0.723449         4
   EC106 Sundays River Valley Sarah Baartman               14823.0                 10763.0        0.273899                      0.726101         5
     BUF         Buffalo City   Buffalo City              238729.0         

In [ ]:
# ============================================================
# 6b. Fix district name mismatches before hybrid merge
# ============================================================

print("District names in merged:")
print(sorted(merged["district"].dropna().unique()))
print()
print("District names in trend:")
print(sorted(trend["district_municipality"].dropna().unique()))

District names in merged:
['Alfred Nzo', 'Amathole', 'Buffalo City', 'Chris Hani', 'Joe Gqabi', 'Nelson Mandela Bay', 'O.R. Tambo', 'Sarah Baartman']

District names in trend:
['Alfred Nzo District', 'Amathole District Municipality', 'Chris Hani', 'Joe Gqabi District', 'OR Tambo District', 'Sarah Baartman']


In [ ]:
# ============================================================
# 6c. Manual mapping for mismatched district names
# ============================================================

district_map = {
    "Amathole District Municipality": "Amathole District",
    "Chris Hani District Municipality": "Chris Hani",
    "Joe Gqabi District Municipality": "Joe Gqabi District",
    "OR Tambo District Municipality": "OR Tambo District",
    "Alfred Nzo District Municipality": "Alfred Nzo District",
    "Sarah Baartman District Municipality": "Sarah Baartman",
}

# Apply only if the exact string is in the map; otherwise keep as-is
merged["district_normalised"] = merged["district"].map(
    lambda x: district_map.get(x, x)
)

# Sanity check
print("Normalised districts not matching trend:")
matched = set(merged["district_normalised"]) & set(trend["district_municipality"])
unmatched = set(merged["district_normalised"]) - set(trend["district_municipality"])
print("  Matched:", sorted(matched))
print("  Unmatched:", sorted(unmatched))

Normalised districts not matching trend:
  Matched: ['Chris Hani', 'Sarah Baartman']
  Unmatched: ['Alfred Nzo', 'Amathole', 'Buffalo City', 'Joe Gqabi', 'Nelson Mandela Bay', 'O.R. Tambo']


In [ ]:
d16 = district[district["year"] == 2016].copy()
d21 = district[district["year"] == 2021].copy()

trend = d16[["district_municipality", "registered_youth_total"]].rename(
    columns={"registered_youth_total": "youth_registered_2016"}
).merge(
    d21[["district_municipality", "registered_youth_total"]].rename(
        columns={"registered_youth_total": "youth_registered_2021"}
    ),
    on="district_municipality",
    how="inner"
)

trend["youth_registration_change_pct"] = (
    (trend["youth_registered_2021"] - trend["youth_registered_2016"])
    / trend["youth_registered_2016"] * 100
)

trend = trend.sort_values("youth_registration_change_pct").reset_index(drop=True)
trend["trend_rank"] = trend.index + 1

print("Districts ranked by youth registration decline (2016 -> 2021):")
print(trend.to_string(index=False))

Districts ranked by youth registration decline (2016 -> 2021):
         district_municipality  youth_registered_2016  youth_registered_2021  youth_registration_change_pct  trend_rank
            Joe Gqabi District               100150.0                86332.0                     -13.797304           1
Amathole District Municipality               189825.0               166792.0                     -12.133807           2
                Sarah Baartman               103908.0                92450.0                     -11.027062           3
             OR Tambo District               367948.0               331206.0                      -9.985650           4
                    Chris Hani                43607.0                39271.0                      -9.943358           5
           Alfred Nzo District               205307.0               198620.0                      -3.257074           6


In [ ]:
# ============================================================
# 7. Hybrid scoring (widest + fastest-growing)
# ============================================================

merged = merged.merge(
    trend[["district_municipality", "youth_registration_change_pct", "trend_rank"]],
    left_on="district_normalised",
    right_on="district_municipality",
    how="left"
)

# For metros (BUF, NMA) and any remaining unmatched, fall back to province average
province_avg = trend["youth_registration_change_pct"].mean()

merged["youth_registration_change_pct"] = merged["youth_registration_change_pct"].fillna(province_avg)

merged["gap_norm"] = (
    (merged["youth_gap_2026"] - merged["youth_gap_2026"].min())
    / (merged["youth_gap_2026"].max() - merged["youth_gap_2026"].min())
)

merged["trend_norm"] = (
    (merged["youth_registration_change_pct"] - merged["youth_registration_change_pct"].min())
    / (merged["youth_registration_change_pct"].max() - merged["youth_registration_change_pct"].min())
)

merged["youth_disengagement_risk"] = (
    0.6 * merged["gap_norm"] + 0.4 * merged["trend_norm"]
)

priority = (
    merged
    .sort_values("youth_disengagement_risk", ascending=False)
    [["mdb_code", "municipality", "district",
      "youth_gap_2026", "youth_registration_change_pct",
      "youth_disengagement_risk"]]
    .reset_index(drop=True)
)

priority["priority_rank"] = priority.index + 1

print("Top 15 priority municipalities:")
print(priority.head(15).to_string(index=False))

Top 15 priority municipalities:
mdb_code              municipality           district  youth_gap_2026  youth_registration_change_pct  youth_disengagement_risk  priority_rank
   EC139             Enoch Mgijima         Chris Hani        0.276551                      -9.943358                  0.885639              1
   EC145             Walter Sisulu          Joe Gqabi        0.291210                     -10.024043                  0.881599              2
     BUF              Buffalo City       Buffalo City        0.255109                     -10.024043                  0.818205              3
   EC124                 Amahlathi           Amathole        0.249797                     -10.024043                  0.808876              4
   EC131           Inxuba Yethemba         Chris Hani        0.221738                      -9.943358                  0.789385              5
     NMA        Nelson Mandela Bay Nelson Mandela Bay        0.146583                     -10.024043                

In [ ]:
merged.to_csv(PROCESSED / "municipality_features.csv", index=False)
widest.to_csv(PROCESSED / "municipality_widest_gap.csv", index=False)
trend.to_csv(PROCESSED / "district_trend.csv", index=False)
priority.to_csv(PROCESSED / "priority_municipalities.csv", index=False)

print("Saved:")
print("  municipality_features.csv     — full feature table")
print("  municipality_widest_gap.csv   — ranked by 2026 gap")
print("  district_trend.csv            — 2016->2021 registration decline")
print("  priority_municipalities.csv   — hybrid priority list")

Saved:
  municipality_features.csv     — full feature table
  municipality_widest_gap.csv   — ranked by 2026 gap
  district_trend.csv            — 2016->2021 registration decline
  priority_municipalities.csv   — hybrid priority list


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path
Path("/content/drive/MyDrive/dirisa-sdc/raw").mkdir(parents=True, exist_ok=True)
Path("/content/drive/MyDrive/dirisa-sdc/processed").mkdir(parents=True, exist_ok=True)
print("Folders created.")

Folders created.


In [ ]:
from pathlib import Path
print("Files in processed folder:")
for f in sorted(Path("/content/drive/MyDrive/dirisa-sdc/processed").glob("*.csv")):
    print("  ", f.name)

Files in processed folder:
   clean_census_population.csv
   clean_district_youth_history.csv
   clean_master_municipality.csv
   clean_provincial_turnout_2021.csv
   clean_youth_registration.csv
   district_trend.csv
   municipality_features.csv
   municipality_widest_gap.csv
   priority_municipalities.csv


In [29]:
print(merged.columns.tolist())

['mdb_code', 'municipality', 'district', 'total_pop_2022', 'youth_pop_20_34_2022', 'youth_share_pct_2022', 'registered_2016', 'valid_votes_2016', 'spoilt_votes_2016', 'votes_cast_2016', 'turnout_pct_2016', 'spoilt_rate_pct_2016', 'registered_2021', 'valid_votes_2021', 'spoilt_votes_2021', 'votes_cast_2021', 'turnout_pct_2021', 'spoilt_rate_pct_2021', 'registered_growth_2016_2021_pct', 'turnout_change_2016_2021_pp', 'registered_youth_total', 'youth_gap_2026', 'youth_registration_rate_2026', 'district_normalised', 'district_municipality', 'youth_registration_change_pct', 'trend_rank', 'gap_norm', 'trend_norm', 'youth_disengagement_risk']
